### RAG PIPELINE (INJESTION + RETRIEVAL)
![Workflow](Images/krish_rag_pipeline.png)


### Here we would be looking at the inference code, how we actually fetch the most relevent chunks from the vectorDB

In [1]:
import os
import uuid
from typing import Any, Dict, List, Tuple
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer

In [2]:
# 1. Embedding Manager Class
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

In [ ]:
# 2. Vector Store Class- This code connects to the existing ChromaDB vector store during inference.
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store",
    ):
        """Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"},
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(
                f"Existing documents in collection: {self.collection.count()}"
            )
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

### No add documents present here since this is at the time of inference, not injestion

In [ ]:
# 3. RAG Retriever Class
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(
        self, vector_store: VectorStore, embedding_manager: EmbeddingManager
    ):
        """Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self, query: str, top_k: int = 5, score_threshold: float = 0.0 # arguments
    ) -> List[Dict[str, Any]]: # returns list of results(strings) and metadata
        """Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0] # use the same model used to embed in injestion time

        # Search in the vector DB
        try:
            # Search in ChromaDB collection
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()], # give query embeddings in form of a list
                n_results=top_k # number if top ranked chunks to fetch
            )

            # Process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances) # zipping is basically trying to create a tuple here
                ):
                    """
                    # Convert distance to similarity score (ChromaDB uses cosine distance as similarity score)
                    # We dont compare our embedding to ALL the chunks present in the vector DB, ANN - Approximate Nearest Neighbour =  general problem/approach: find nearest vectors approximately without comparing against every vector.
                    # HNSW - Hierarchical Navigable Small World and IVF = Inverted File Index = a specific algorithm/data structure used to solve that ANN problem efficiently. 
                    """

                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append(
                            {
                                "id": doc_id,
                                "content": document,
                                "metadata": metadata,
                                "similarity_score": similarity_score,
                                "distance": distance,
                                "rank": i + 1,
                            }
                        )

                print(
                    f"Retrieved {len(retrieved_docs)} documents (after filtering)"
                )
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

In [9]:
# Initialize embedding manager
embedding_manager = EmbeddingManager(model_name="all-MiniLM-L6-v2")

# Connect to your existing saved vector database
# (Verify relative path ../data/vector_store matches where your first notebook created the database)
vector_store = VectorStore(
    collection_name="pdf_documents",
    persist_directory="data/vector_store"
)

# Initialize retriever
retriever = RAGRetriever(
    vector_store=vector_store,
    embedding_manager=embedding_manager
)

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

C:\Users\Nitin Phanse\AppData\Local\Temp\ipykernel_5168\1182172884.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}"


Model loaded successfully. Embedding dimension: 384
Vector store initialized. Collection: pdf_documents
Existing documents in collection: 1020


In [10]:
query = "What is attention is all you need"

# Retrieve top 3 relevant chunks
results = retriever.retrieve(query=query, top_k=3, score_threshold=0.0)

# Display retrieved chunks
for doc in results:
    print(f"\n--- Rank {doc['rank']} | Score: {doc['similarity_score']:.4f} ---")
    print(f"Content: {doc['content']}")
    print(f"Metadata: {doc['metadata']}")

Retrieving documents for query: 'What is attention is all you need'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)

--- Rank 1 | Score: 0.1400 ---
Content: 3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3
Metadata: {'file_type': 'pdf', 'keywords': '', 'creator': 'LaTeX with hyperref', 'author': '', 'doc_index': 165, 'total_pages': 15, 'subject': '', 'page_label': '3', 'title': '', 'source': 'data\\pdf\\attention is all you need.pdf', 'content_length': 216, 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source_file': 'attention is all you need.pdf', 'producer': 'pdfTeX-1.40.25', 'page': 2, 'moddate': '2024-04-10T21:11:43+00:00', 'trapped': '/False', 'creationdate': '2024-04-10T21:11:43+00:00'}


### What we get above is the chunks, which basically acts as a context for the llm